# int8 quantization, by hand and then properly

The scale-and-unscale trick that makes int8 matmul lossless enough — implemented from scratch, then applied with one method call.

**Runs on:** CPU — about 4 minutes &nbsp;·&nbsp; **Slides:** [Chapter 18 — Best Practices for the Real World](../../../course-web-slides/ch18/index.html) &nbsp;·&nbsp; **Section:** 04 — Faster inference with quantization

---

## Why casting naively fails

In [ ]:
from keras import ops
import numpy as np

x = ops.array([[0.1, 0.9], [1.2, -0.8]])
kernel = ops.array([[-0.1, -2.2], [1.1, 0.7]])

print("naive cast to int8:")
print(ops.convert_to_numpy(ops.cast(x, "int8")))
print()
print("Everything below 1.0 becomes zero. Total loss of information.")

## abs-max scaling

In [ ]:
def abs_max_quantize(value):
    abs_max = ops.max(ops.abs(value), keepdims=True)
    scale = ops.divide(127, abs_max + 1e-7)
    scaled_value = value * scale
    scaled_value = ops.clip(ops.round(scaled_value), -127, 127)
    scaled_value = ops.cast(scaled_value, dtype="int8")
    return scaled_value, scale

int_x, x_scale = abs_max_quantize(x)
int_kernel, kernel_scale = abs_max_quantize(kernel)

print("x as int8:     ", ops.convert_to_numpy(int_x))
print("kernel as int8:", ops.convert_to_numpy(int_kernel))
print(f"scales: {float(x_scale):.2f}, {float(kernel_scale):.2f}")

The tensor is spread across the full **[-127, 127]** range before casting. `+ 1e-7` avoids dividing by zero, and **rounding and clipping before the cast** is more accurate than casting directly.

## The matmul, and unscaling

In [ ]:
int_y = ops.matmul(int_x, int_kernel)
y = ops.cast(int_y, dtype="float32") / (x_scale * kernel_scale)

print("quantized result:")
print(ops.convert_to_numpy(y))
print()
print("float32 result:")
print(ops.convert_to_numpy(ops.matmul(x, kernel)))
print()
err = np.abs(ops.convert_to_numpy(y) - ops.convert_to_numpy(ops.matmul(x, kernel)))
print(f"max absolute error: {err.max():.4f}")

**matmul is linear**, so the final unscaling cancels the initial scaling exactly. Any error comes **only from the rounding** when casting to int8 — not from the multiplication.

The added operations are abs, max, clip, cast, divide, multiply — all elementwise and fast, against a matmul that is now int8 and can be considerably faster than even float16.

## How the error scales with matrix size

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
sizes = [8, 32, 128, 512]
errs = []
for n in sizes:
    a = ops.array(rng.normal(size=(n, n)).astype("float32"))
    b = ops.array(rng.normal(size=(n, n)).astype("float32"))
    ia, sa = abs_max_quantize(a)
    ib, sb = abs_max_quantize(b)
    q = ops.cast(ops.matmul(ia, ib), "float32") / (sa * sb)
    f = ops.matmul(a, b)
    rel = np.abs(ops.convert_to_numpy(q - f)).mean() / np.abs(
        ops.convert_to_numpy(f)).mean()
    errs.append(rel)
    print(f"{n:4d}x{n:<4d} mean relative error {rel:.4f}")

plt.figure(figsize=(6, 3.8))
plt.semilogx(sizes, errs, "o-")
plt.xlabel("matrix size"); plt.ylabel("mean relative error")
plt.title("Quantization error stays small as the matmul grows")
plt.show()

The error does not blow up with size, because the rounding errors are independent and largely cancel in the sum. **That is what makes the technique usable on real models** rather than only on 2×2 examples.

## One method call

In [ ]:
import keras, os
from keras import layers

(x_tr, y_tr), (xt, yt) = keras.datasets.mnist.load_data()
x_tr = x_tr.reshape(-1, 784).astype("float32") / 255
xt = xt.reshape(-1, 784).astype("float32") / 255

keras.utils.set_random_seed(0)
model = keras.Sequential([layers.Dense(512, activation="relu"),
                          layers.Dense(256, activation="relu"),
                          layers.Dense(10, activation="softmax")])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.fit(x_tr, y_tr, epochs=4, batch_size=128, verbose=0)
model.save("fp32.keras")
base_acc = model.evaluate(xt, yt, verbose=0)[1]

q = keras.saving.load_model("fp32.keras")
q.quantize("int8")
q.save("int8.keras")
q_acc = q.evaluate(xt, yt, verbose=0)[1]

fp = os.path.getsize("fp32.keras") / 1e6
qs = os.path.getsize("int8.keras") / 1e6
print(f"float32: {fp:6.2f} MB   accuracy {base_acc:.4f}")
print(f"int8:    {qs:6.2f} MB   accuracy {q_acc:.4f}")
print(f"\n{fp/qs:.1f}x smaller, accuracy cost {base_acc - q_acc:+.4f}")

> ⚠️ **`quantize()` converts the weights **in place**.** It is a one-way operation on that model object — keep the float32 file, as done above.

## Where the error lands

In [ ]:
import numpy as np

p32 = model.predict(xt, verbose=0)
p8 = q.predict(xt, verbose=0)

disagree = (p32.argmax(1) != p8.argmax(1))
print(f"{disagree.sum()} of {len(xt)} predictions changed "
      f"({disagree.mean():.2%})")

conf32 = p32.max(1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.hist(conf32[~disagree], bins=40, alpha=.7, label="unchanged", density=True)
a1.hist(conf32[disagree], bins=40, alpha=.7, label="changed", density=True)
a1.set_xlabel("float32 confidence"); a1.legend()
a1.set_title("Disagreements cluster at LOW confidence")

a2.scatter(p32.max(1), p8.max(1), s=2, alpha=.2)
a2.plot([0, 1], [0, 1], "k--", lw=1)
a2.set_xlabel("float32 confidence"); a2.set_ylabel("int8 confidence")
a2.set_title("Confidence is largely preserved")
plt.tight_layout(); plt.show()

**The predictions that change are the ones the model was unsure about anyway.** That is the reassuring shape — but measure it on *your* data. It is not guaranteed, and a model whose high-confidence predictions move under quantization is telling you something.

## Which layers it covers

In [ ]:
print("int8 quantization is built into:")
print("  Dense, EinsumDense, Embedding")
print()
print("EinsumDense is what MultiHeadAttention uses -- which means")
print("int8 inference works for any Transformer-based model.")
print()
print("Convolutions are not covered by model.quantize(). For vision")
print("models, look at post-training quantization in the deployment")
print("runtime you are targeting.")

---

## What to take away

- Scale into [-127, 127], cast, multiply, unscale — matmul is linear so the scaling cancels.
- Rounding is the only source of error, and it does not grow with matrix size.
- `model.quantize("int8")` is one line and **in place** — keep the float32 file.
- Disagreements cluster at low confidence, but verify that on your own data.